In [ ]:
import argparse
import time
import yaml
import os
import logging
from collections import OrderedDict
from contextlib import suppress
from datetime import datetime
from spikingjelly.clock_driven import functional

import torch
import torch.nn as nn
import torchvision.utils
from torch.nn.parallel import DistributedDataParallel as NativeDDP

from timm.data import create_dataset, resolve_data_config, Mixup, FastCollateMixup, AugMixDataset
from timm.models import create_model, safe_model_name, resume_checkpoint, load_checkpoint, \
     model_parameters
from timm.models.layers import convert_splitbn_model
from timm.utils import *
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy, JsdCrossEntropy
from timm.optim import create_optimizer_v2, optimizer_kwargs
from timm.scheduler import create_scheduler
from timm.utils import ApexScaler, NativeScaler
import model
try:
    from apex import amp
    from apex.parallel import DistributedDataParallel as ApexDDP
    from apex.parallel import convert_syncbn_model

    has_apex = True
except ImportError:
    has_apex = False

has_native_amp = False
try:
    if getattr(torch.cuda.amp, 'autocast') is not None:
        has_native_amp = True
except AttributeError:
    pass

try:
    import wandb

    has_wandb = True
except ImportError:
    has_wandb = False

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

In [3]:
torch.backends.cudnn.benchmark = True
_logger = logging.getLogger('train')

In [4]:
# The first arg parser parses out only the --config argument, this argument is used to
# load a yaml file containing key-values that override the defaults for the main parser below
config_parser = parser = argparse.ArgumentParser(description='Training Config', add_help=False)
parser.add_argument('-c', '--config', default='shd_cleanup.yml', type=str, metavar='FILE',
                    help='YAML config file specifying default arguments')

parser = argparse.ArgumentParser(description='PyTorch ImageNet Training')

# Model detail
parser.add_argument('--model', default='sdt', type=str, metavar='MODEL',
                    help='Name of model to train (default: "countception"')
parser.add_argument('-T', '--time-step', type=int, default=4, metavar='time',
                    help='simulation time step of spiking neuron (default: 4)')
parser.add_argument('-L', '--layer', type=int, default=4, metavar='layer',
                    help='model layer (default: 4)')
parser.add_argument('--num-classes', type=int, default=None, metavar='N',
                    help='number of label classes (Model default if None)')
parser.add_argument('--img-size', type=int, default=None, metavar='N',
                    help='Image patch size (default: None => model default)')
parser.add_argument('--input-size', default=None, nargs=3, type=int,
                    metavar='N N N',
                    help='input all input dimensions (d h w, e.g. --input-size 3 224 224), uses model default if empty')
parser.add_argument('--dim', type=int, default=None, metavar='N',
                    help='embedding dimsension of feature')
parser.add_argument('--num_heads', type=int, default=None, metavar='N',
                    help='attention head number')
parser.add_argument('--patch-size', type=int, default=None, metavar='N',
                    help='Image patch size')
parser.add_argument('--mlp-ratio', type=int, default=None, metavar='N',
                    help='expand ration of embedding dimension in MLP block')
# Dataset / Model parameters
parser.add_argument('-data-dir', metavar='DIR',default="/home/zhou/Compact-Transformers-main/cifar-10-python/",
                    help='path to dataset')
parser.add_argument('--dataset', '-d', metavar='NAME', default='torch/cifar10',
                    help='dataset type (default: ImageFolder/ImageTar if empty)')
parser.add_argument('--train-split', metavar='NAME', default='train',
                    help='dataset train split (default: train)')
parser.add_argument('--val-split', metavar='NAME', default='validation',
                    help='dataset validation split (default: validation)')
parser.add_argument('--pretrained', action='store_true', default=False,
                    help='Start with pretrained version of specified network (if avail)')
parser.add_argument('--initial-checkpoint', default='', type=str, metavar='PATH',
                    help='Initialize model from this checkpoint (default: none)')
parser.add_argument('--resume', default='', type=str, metavar='PATH',
                    help='Resume full model and optimizer state from checkpoint (default: none)')
parser.add_argument('--no-resume-opt', action='store_true', default=False,
                    help='prevent resume of optimizer state when resuming model')

parser.add_argument('--gp', default=None, type=str, metavar='POOL',
                    help='Global pool type, one of (fast, avg, max, avgmax, avgmaxc). Model default if None.')

parser.add_argument('--crop-pct', default=None, type=float,
                    metavar='N', help='image image center crop percent (for validation only)')
parser.add_argument('--mean', type=float, nargs='+', default=None, metavar='MEAN',
                    help='Override mean pixel value of dataset')
parser.add_argument('--std', type=float, nargs='+', default=None, metavar='STD',
                    help='Override std deviation of of dataset')
parser.add_argument('--interpolation', default='', type=str, metavar='NAME',
                    help='Image resize interpolation type (overrides model)')
parser.add_argument('-b', '--batch-size', type=int, default=32, metavar='N',
                    help='image batch size for training (default: 32)')
parser.add_argument('-vb', '--val-batch-size', type=int, default=32, metavar='N',
                    help='image val batch size for training (default: 32)')
# parser.add_argument('-vb', '--validation-batch-size-multiplier', type=int, default=1, metavar='N',
#                     help='ratio of validation batch size to training batch size (default: 1)')

# Optimizer parameters
parser.add_argument('--opt', default='sgd', type=str, metavar='OPTIMIZER',
                    help='Optimizer (default: "sgd"')
parser.add_argument('--opt-eps', default=None, type=float, metavar='EPSILON',
                    help='Optimizer Epsilon (default: None, use opt default)')
parser.add_argument('--opt-betas', default=None, type=float, nargs='+', metavar='BETA',
                    help='Optimizer Betas (default: None, use opt default)')
parser.add_argument('--momentum', type=float, default=0.9, metavar='M',
                    help='Optimizer momentum (default: 0.9)')
parser.add_argument('--weight-decay', type=float, default=0.0001,
                    help='weight decay (default: 0.0001)')
parser.add_argument('--clip-grad', type=float, default=None, metavar='NORM',
                    help='Clip gradient norm (default: None, no clipping)')
parser.add_argument('--clip-mode', type=str, default='norm',
                    help='Gradient clipping mode. One of ("norm", "value", "agc")')

# Learning rate schedule parameters
parser.add_argument('--sched', default='step', type=str, metavar='SCHEDULER',
                    help='LR scheduler (default: "step"')
parser.add_argument('--lr', type=float, default=0.01, metavar='LR',
                    help='learning rate (default: 0.01)')
parser.add_argument('--lr-noise', type=float, nargs='+', default=None, metavar='pct, pct',
                    help='learning rate noise on/off epoch percentages')
parser.add_argument('--lr-noise-pct', type=float, default=0.67, metavar='PERCENT',
                    help='learning rate noise limit percent (default: 0.67)')
parser.add_argument('--lr-noise-std', type=float, default=1.0, metavar='STDDEV',
                    help='learning rate noise std-dev (default: 1.0)')
parser.add_argument('--lr-cycle-mul', type=float, default=1.0, metavar='MULT',
                    help='learning rate cycle len multiplier (default: 1.0)')
parser.add_argument('--lr-cycle-limit', type=int, default=1, metavar='N',
                    help='learning rate cycle limit')
parser.add_argument('--warmup-lr', type=float, default=0.0001, metavar='LR',
                    help='warmup learning rate (default: 0.0001)')
parser.add_argument('--min-lr', type=float, default=1e-5, metavar='LR',
                    help='lower lr bound for cyclic schedulers that hit 0 (1e-5)')
parser.add_argument('--epochs', type=int, default=200, metavar='N',
                    help='number of epochs to train (default: 2)')
parser.add_argument('--epoch-repeats', type=float, default=0., metavar='N',
                    help='epoch repeat multiplier (number of times to repeat dataset epoch per train epoch).')
parser.add_argument('--start-epoch', default=None, type=int, metavar='N',
                    help='manual epoch number (useful on restarts)')
parser.add_argument('--decay-epochs', type=float, default=30, metavar='N',
                    help='epoch interval to decay LR')
parser.add_argument('--warmup-epochs', type=int, default=3, metavar='N',
                    help='epochs to warmup LR, if scheduler supports')
parser.add_argument('--cooldown-epochs', type=int, default=10, metavar='N',
                    help='epochs to cooldown LR at min_lr, after cyclic schedule ends')
parser.add_argument('--patience-epochs', type=int, default=10, metavar='N',
                    help='patience epochs for Plateau LR scheduler (default: 10')
parser.add_argument('--decay-rate', '--dr', type=float, default=0.1, metavar='RATE',
                    help='LR decay rate (default: 0.1)')

# Augmentation & regularization parameters
parser.add_argument('--no-aug', action='store_true', default=False,
                    help='Disable all training augmentation, override other train aug args')
parser.add_argument('--scale', type=float, nargs='+', default=[0.08, 1.0], metavar='PCT',
                    help='Random resize scale (default: 0.08 1.0)')
parser.add_argument('--ratio', type=float, nargs='+', default=[1.0, 1.0], metavar='RATIO',
                    help='Random resize aspect ratio (default: 0.75 1.33)')
parser.add_argument('--hflip', type=float, default=0.5,
                    help='Horizontal flip training aug probability')
parser.add_argument('--vflip', type=float, default=0.,
                    help='Vertical flip training aug probability')
parser.add_argument('--color-jitter', type=float, default=0.4, metavar='PCT',
                    help='Color jitter factor (default: 0.4)')
parser.add_argument('--aa', type=str, default=None, metavar='NAME',
                    help='Use AutoAugment policy. "v0" or "original". (default: None)'),
parser.add_argument('--aug-splits', type=int, default=0,
                    help='Number of augmentation splits (default: 0, valid: 0 or >=2)')
parser.add_argument('--jsd', action='store_true', default=False,
                    help='Enable Jensen-Shannon Divergence + CE loss. Use with `--aug-splits`.')
parser.add_argument('--reprob', type=float, default=0., metavar='PCT',
                    help='Random erase prob (default: 0.)')
parser.add_argument('--remode', type=str, default='const',
                    help='Random erase mode (default: "const")')
parser.add_argument('--recount', type=int, default=1,
                    help='Random erase count (default: 1)')
parser.add_argument('--resplit', action='store_true', default=False,
                    help='Do not random erase first (clean) augmentation split')
parser.add_argument('--mixup', type=float, default=0.0,
                    help='mixup alpha, mixup enabled if > 0. (default: 0.)')
parser.add_argument('--cutmix', type=float, default=0.0,
                    help='cutmix alpha, cutmix enabled if > 0. (default: 0.)')
parser.add_argument('--cutmix-minmax', type=float, nargs='+', default=None,
                    help='cutmix min/max ratio, overrides alpha and enables cutmix if set (default: None)')
parser.add_argument('--mixup-prob', type=float, default=1.0,
                    help='Probability of performing mixup or cutmix when either/both is enabled')
parser.add_argument('--mixup-switch-prob', type=float, default=0.5,
                    help='Probability of switching to cutmix when both mixup and cutmix enabled')
parser.add_argument('--mixup-mode', type=str, default='batch',
                    help='How to apply mixup/cutmix params. Per "batch", "pair", or "elem"')
parser.add_argument('--mixup-off-epoch', default=0, type=int, metavar='N',
                    help='Turn off mixup after this epoch, disabled if 0 (default: 0)')
parser.add_argument('--smoothing', type=float, default=0.1,
                    help='Label smoothing (default: 0.1)')
parser.add_argument('--train-interpolation', type=str, default='random',
                    help='Training interpolation (random, bilinear, bicubic default: "random")')
parser.add_argument('--drop', type=float, default=0.0, metavar='PCT',
                    help='Dropout rate (default: 0.)')
parser.add_argument('--drop-connect', type=float, default=None, metavar='PCT',
                    help='Drop connect rate, DEPRECATED, use drop-path (default: None)')
parser.add_argument('--drop-path', type=float, default=None, metavar='PCT',
                    help='Drop path rate (default: None)')
parser.add_argument('--drop-block', type=float, default=None, metavar='PCT',
                    help='Drop block rate (default: None)')

# Batch norm parameters (only works with gen_efficientnet based models currently)
parser.add_argument('--bn-tf', action='store_true', default=False,
                    help='Use Tensorflow BatchNorm defaults for models that support it (default: False)')
parser.add_argument('--bn-momentum', type=float, default=None,
                    help='BatchNorm momentum override (if not None)')
parser.add_argument('--bn-eps', type=float, default=None,
                    help='BatchNorm epsilon override (if not None)')
parser.add_argument('--sync-bn', action='store_true',
                    help='Enable NVIDIA Apex or Torch synchronized BatchNorm.')
parser.add_argument('--dist-bn', type=str, default='',
                    help='Distribute BatchNorm stats between nodes after each epoch ("broadcast", "reduce", or "")')
parser.add_argument('--split-bn', action='store_true',
                    help='Enable separate BN layers per augmentation split.')

# Model Exponential Moving Average
parser.add_argument('--model-ema', action='store_true', default=False,
                    help='Enable tracking moving average of model weights')
parser.add_argument('--model-ema-force-cpu', action='store_true', default=False,
                    help='Force ema to be tracked on CPU, rank=0 node only. Disables EMA validation.')
parser.add_argument('--model-ema-decay', type=float, default=0.9998,
                    help='decay factor for model weights moving average (default: 0.9998)')

# Misc
parser.add_argument('--seed', type=int, default=42, metavar='S',
                    help='random seed (default: 42)')
parser.add_argument('--log-interval', type=int, default=1000, metavar='N',
                    help='how many batches to wait before logging training status')
parser.add_argument('--recovery-interval', type=int, default=0, metavar='N',
                    help='how many batches to wait before writing recovery checkpoint')
parser.add_argument('--checkpoint-hist', type=int, default=10, metavar='N',
                    help='number of checkpoints to keep (default: 10)')
parser.add_argument('-j', '--workers', type=int, default=4, metavar='N',
                    help='how many training processes to use (default: 1)')
parser.add_argument('--save-images', action='store_true', default=False,
                    help='save images of image bathes every log interval for debugging')
parser.add_argument('--amp', action='store_true', default=False,
                    help='use NVIDIA Apex AMP or Native AMP for mixed precision training')
parser.add_argument('--apex-amp', action='store_true', default=False,
                    help='Use NVIDIA Apex AMP mixed precision')
parser.add_argument('--native-amp', action='store_true', default=False,
                    help='Use Native Torch AMP mixed precision')
parser.add_argument('--channels-last', action='store_true', default=False,
                    help='Use channels_last memory layout')
parser.add_argument('--pin-mem', action='store_true', default=False,
                    help='Pin CPU memory in DataLoader for more efficient (sometimes) transfer to GPU.')
parser.add_argument('--no-prefetcher', action='store_true', default=False,
                    help='disable fast prefetcher')
parser.add_argument('--output', default='', type=str, metavar='PATH',
                    help='path to output folder (default: none, current dir)')
parser.add_argument('--experiment', default='', type=str, metavar='NAME',
                    help='name of train experiment, name of sub-folder for output')
parser.add_argument('--eval-metric', default='top1', type=str, metavar='EVAL_METRIC',
                    help='Best metric (default: "top1"')
parser.add_argument('--tta', type=int, default=0, metavar='N',
                    help='Test/inference time augmentation (oversampling) factor. 0=None (default: 0)')
parser.add_argument("--local_rank", default=0, type=int)
parser.add_argument('--use-multi-epochs-loader', action='store_true', default=False,
                    help='use the multi-epochs-loader to save time at the beginning of every epoch')
parser.add_argument('--torchscript', dest='torchscript', action='store_true',
                    help='convert model torchscript for inference')
parser.add_argument('--log-wandb', action='store_true', default=False,
                    help='log training and validation metrics to wandb')
parser.add_argument("-f", "--fff", help="a dummy argument to fool ipython", default="1")

def _parse_args():
    # Do we have a config file to parse?
    args_config, remaining = config_parser.parse_known_args()
    args_config.config = "shd_cleanup_copy.yml"
    if args_config.config:
        with open(args_config.config, 'r') as f:
            cfg = yaml.safe_load(f)
            parser.set_defaults(**cfg)

    # The main arg parser parses the rest of the args, the usual
    # defaults will have been overridden if config file specified.
    args = parser.parse_args(remaining)

    # Cache the args as a text string to save them in the output dir later
    args_text = yaml.safe_dump(args.__dict__, default_flow_style=False)
    return args, args_text

In [5]:
def train_one_epoch(
        epoch, model, loader, optimizer, loss_fn, args,
        lr_scheduler=None, saver=None, output_dir=None, amp_autocast=suppress,
        loss_scaler=None, model_ema=None, mixup_fn=None):
    if args.mixup_off_epoch and epoch >= args.mixup_off_epoch:
        if args.prefetcher and loader.mixup_enabled:
            loader.mixup_enabled = False
        elif mixup_fn is not None:
            mixup_fn.mixup_enabled = False

    second_order = hasattr(optimizer, 'is_second_order') and optimizer.is_second_order
    batch_time_m = AverageMeter()
    data_time_m = AverageMeter()
    losses_m = AverageMeter()

    model.cuda()
    model.train()

    end = time.time()
    last_idx = len(loader) - 1
    num_updates = epoch * len(loader)
    for batch_idx, y in enumerate(loader):
        image = y['image']
        audio = y['audio']
        target = y['label']
        last_batch = batch_idx == last_idx
        data_time_m.update(time.time() - end)
        image, audio, target = image.cuda(), audio.cuda(), target.cuda()

        with amp_autocast():
            output = model(image, audio)
            output.cuda()
            loss = loss_fn(output, target)
            loss.cuda()

        if not args.distributed:
            losses_m.update(loss.item(), image.size(0))

        optimizer.zero_grad()
        if loss_scaler is not None:
            loss_scaler(
                loss, optimizer,
                clip_grad=args.clip_grad, clip_mode=args.clip_mode,
                parameters=model_parameters(model, exclude_head='agc' in args.clip_mode),
                create_graph=second_order)
        else:
            # loss.backward()
            loss.backward(create_graph=second_order)
            if args.clip_grad is not None:
                dispatch_clip_grad(
                    model_parameters(model, exclude_head='agc' in args.clip_mode),
                    value=args.clip_grad, mode=args.clip_mode)
            optimizer.step()

        functional.reset_net(model)

        if model_ema is not None:
            model_ema.update(model)

        torch.cuda.synchronize()
        num_updates += 1
        batch_time_m.update(time.time() - end)
        if last_batch or batch_idx % args.log_interval == 0:
            lrl = [param_group['lr'] for param_group in optimizer.param_groups]
            lr = sum(lrl) / len(lrl)

            if args.distributed:
                reduced_loss = reduce_tensor(loss.data, args.world_size)
                losses_m.update(reduced_loss.item(), image.size(0))

            if args.local_rank == 0:
                _logger.info(
                    'Train: {} [{:>4d}/{} ({:>3.0f}%)]  '
                    'Loss: {loss.val:>9.6f} ({loss.avg:>6.4f})  '
                    'Time: {batch_time.val:.3f}s, {rate:>7.2f}/s  '
                    '({batch_time.avg:.3f}s, {rate_avg:>7.2f}/s)  '
                    'LR: {lr:.3e}  '
                    'Data: {data_time.val:.3f} ({data_time.avg:.3f})'.format(
                        epoch,
                        batch_idx, len(loader),
                        100. * batch_idx / last_idx,
                        loss=losses_m,
                        batch_time=batch_time_m,
                        rate=image.size(0) * args.world_size / batch_time_m.val,
                        rate_avg=image.size(0) * args.world_size / batch_time_m.avg,
                        lr=lr,
                        data_time=data_time_m))

                if args.save_images and output_dir:
                    torchvision.utils.save_image(
                        image,
                        os.path.join(output_dir, 'train-batch-%d.jpg' % batch_idx),
                        padding=0,
                        normalize=True)

        if saver is not None and args.recovery_interval and (
                last_batch or (batch_idx + 1) % args.recovery_interval == 0):
            saver.save_recovery(epoch, batch_idx=batch_idx)

        if lr_scheduler is not None:
            lr_scheduler.step_update(num_updates=num_updates, metric=losses_m.avg)

        end = time.time()
        # end for

    if hasattr(optimizer, 'sync_lookahead'):
        optimizer.sync_lookahead()

    return OrderedDict([('loss', losses_m.avg)])

In [6]:
def validate_multimodal(model, loader, loss_fn, args, amp_autocast=suppress, log_suffix=''):
    batch_time_m = AverageMeter()
    losses_m = AverageMeter()
    top1_m = AverageMeter()
    top5_m = AverageMeter()

    model.cuda()
    model.eval()

    end = time.time()
    last_idx = len(loader) - 1
    with torch.no_grad():
        for batch_idx, y in enumerate(loader):
            image = y['image']
            audio = y['audio']
            target = y['label']
            last_batch = batch_idx == last_idx
            image, audio, target = image.cuda(), audio.cuda(), target.cuda()
            if args.channels_last:
                input = input.contiguous(memory_format=torch.channels_last)

            with amp_autocast:
                output = model(image, audio)
            if isinstance(output, (tuple, list)):
                output = output[0]

            # augmentation reduction
            reduce_factor = args.tta
            if reduce_factor > 1:
                output = output.unfold(0, reduce_factor, reduce_factor).mean(dim=2)
                target = target[0:target.size(0):reduce_factor]
            loss = loss_fn(output, target)
            functional.reset_net(model)

            acc1, acc5 = accuracy(output, target, topk=(1, 5))

            if args.distributed:
                reduced_loss = reduce_tensor(loss.data, args.world_size)
                acc1 = reduce_tensor(acc1, args.world_size)
                acc5 = reduce_tensor(acc5, args.world_size)
            else:
                reduced_loss = loss.data

            torch.cuda.synchronize()

            losses_m.update(reduced_loss.item(), image.size(0))
            top1_m.update(acc1.item(), output.size(0))
            top5_m.update(acc5.item(), output.size(0))

            batch_time_m.update(time.time() - end)
            end = time.time()
            if args.local_rank == 0 and (last_batch or batch_idx % args.log_interval == 0):
                log_name = 'Test' + log_suffix
                _logger.info(
                    '{0}: [{1:>4d}/{2}]  '
                    'Time: {batch_time.val:.3f} ({batch_time.avg:.3f})  '
                    'Loss: {loss.val:>7.4f} ({loss.avg:>6.4f})  '
                    'Acc@1: {top1.val:>7.4f} ({top1.avg:>7.4f})  '
                    'Acc@5: {top5.val:>7.4f} ({top5.avg:>7.4f})'.format(
                        log_name, batch_idx, last_idx, batch_time=batch_time_m,
                        loss=losses_m, top1=top1_m, top5=top5_m))

    metrics = OrderedDict([('loss', losses_m.avg), ('top1', top1_m.avg), ('top5', top5_m.avg)])

    return metrics

In [7]:
setup_default_logging()
args, args_text = _parse_args()

if args.log_wandb:
    if has_wandb:
        wandb.init(project=args.experiment, config=args)
    else:
        _logger.warning("You've requested to log metrics to wandb but package not found. "
                        "Metrics not being logged to wandb, try `pip install wandb`")

args.prefetcher = not args.no_prefetcher
args.distributed = False
if 'WORLD_SIZE' in os.environ:
    args.distributed = int(os.environ['WORLD_SIZE']) > 1
args.device = 'cuda:1'
args.world_size = 1
args.rank = 0  # global rank
if args.distributed:
    args.device = 'cuda:%d' % args.local_rank
    torch.cuda.set_device(args.local_rank)
    torch.distributed.init_process_group(backend='nccl', init_method='env://')
    args.world_size = torch.distributed.get_world_size()
    args.rank = torch.distributed.get_rank()
    _logger.info('Training in distributed mode with multiple processes, 1 GPU per process. Process %d, total %d.'
                    % (args.rank, args.world_size))
else:
    _logger.info('Training with a single process on 1 GPUs.')
assert args.rank >= 0

# resolve AMP arguments based on PyTorch / Apex availability
use_amp = None
if args.amp:
    # `--amp` chooses native amp before apex (APEX ver not actively maintained)
    if has_native_amp:
        args.native_amp = True
    elif has_apex:
        args.apex_amp = True
if args.apex_amp and has_apex:
    use_amp = 'apex'
elif args.native_amp and has_native_amp:
    use_amp = 'native'
elif args.apex_amp or args.native_amp:
    _logger.warning("Neither APEX or native Torch AMP is available, using float32. "
                    "Install NVIDA apex or upgrade to PyTorch 1.6")
random_seed(args.seed, args.rank)

INFO:train:Training with a single process on 1 GPUs.
Training with a single process on 1 GPUs.


In [8]:
import spiking_audio_datasets
shd_train_dataset, shd_test_dataset = spiking_audio_datasets.get_spiking_audio_datasets_just_dataset(
    dataset=args.dataset,
    dir=args.data_dir, 
    batch_size=args.batch_size,
    n_time_bins=args.n_time_bins,
    n_freq_bins=args.n_freq_bins,
    no_aug=args.no_aug,
    time_jitter=args.time_jitter,
    spatial_jitter=args.spatial_jitter,
    max_drop_chunk=args.max_drop_chunk,
    noise=args.noise,
    drop_event=args.drop_event,
    time_skew=args.time_skew,
    cut_mix=args.cut_mix,
    sa_portion=args.sa_portion,
    sa_times=args.sa_times,
    fp16=args.amp,
    num_workers=args.workers,
    pin_memory=True
)


dataset input_mothod="sum_bilinear"
nearest_multiple(n_freq_bins=64, sensor_size[0]=700)=704
train_transform Compose(
    DropEvent(p=0.1)
    TimeSkew(coefficient=(0.8333333333333334, 1.2), offset=0)
    TimeJitter(std=1, clip_negative=False, sort_timestamps=True)
    UniformNoise(sensor_size=(700, 1, 1), n=(0, 35))
    ToFrame(sensor_size=(700, 1, 1), time_window=None, event_count=None, n_time_bins=64, n_event_bins=None, overlap=0, include_incomplete=False)
    ToTensor()
    Lambda()
    Resize(size=[704, 64], interpolation=bilinear, max_size=None, antialias=None)
    Lambda()
    Lambda()
    Lambda()
    FrequencyMasking()
    TimeMasking()
    FrequencyMasking()
    TimeMasking()
    Lambda()
)
test_transform Compose(
    ToFrame(sensor_size=(700, 1, 1), time_window=None, event_count=None, n_time_bins=64, n_event_bins=None, overlap=0, include_incomplete=False)
    ToTensor()
    Lambda()
    Resize(size=[704, 64], interpolation=bilinear, max_size=None, antialias=None)
    Lambda(

In [9]:
from torchvision import transforms
from torchvision.datasets import MNIST

mnist_mean, mnist_std = (0.1307,), (0.3081,)  # for 0..1 tensors

transform_1ch = transforms.Compose([
    transforms.ToTensor(),                         # (1,28,28) in [0,1]
    transforms.Normalize(mean=mnist_mean, std=mnist_std),
])

mnist_train_dataset = MNIST('/dataset/MNIST', train=True,  download=False, transform=transform_1ch)
mnist_test_dataset  = MNIST('/dataset/MNIST', train=False, download=False, transform=transform_1ch)

In [10]:
import torch, random
from torch.utils.data import Dataset, DataLoader

def _coerce_dataset(ds_or_loader):
    # Prefer dataset if a loader was given
    if isinstance(ds_or_loader, DataLoader):
        return ds_or_loader.dataset
    return ds_or_loader

def _enumerate_labels(ds, map_fn, num_classes=10):
    """
    Enumerate the dataset indexing-style: ds[i] -> (x,y) or dict with 'label'.
    Returns:
      labels: list[int], length=len(ds), each in [0,num_classes) (after map_fn)
      buckets: list[list[int]] where buckets[d] = indices with label d
    """
    N = len(ds)
    labels = [None] * N
    buckets = [[] for _ in range(num_classes)]

    for i in range(N):
        sample = ds[i]
        # Allow (x,y) tuple or dicts like {'label': y, ...}
        if isinstance(sample, dict):
            y = sample.get('label')
        else:
            _, y = sample
        if torch.is_tensor(y):
            y = int(y.item())
        else:
            y = int(y)

        d = int(map_fn(y))
        labels[i] = d
        if 0 <= d < num_classes:
            buckets[d].append(i)
    return labels, buckets

class LabelMatchedPairDataset(Dataset):
    """
    Pairs items from two datasets by digit label (0..9).
    - ds_a: anchor dataset (e.g., MNIST). Determines length.
    - ds_b: secondary dataset (e.g., SHD).
    - map_a: map raw label -> digit (MNIST: identity).
    - map_b: map raw label -> digit (SHD: lambda y: y % 10).
    - drop_missing: if True, drop anchor samples whose digit doesn't exist in ds_b.
                    if False, raise an error when a digit is missing.
    - randomized: pick random partner from bucket; else cycle deterministically.
    """
    def __init__(self, ds_a, ds_b, map_a=lambda y: y, map_b=lambda y: y,
                 num_classes=10, drop_missing=False, randomized=True, seed=42):
        self.ds_a = _coerce_dataset(ds_a)
        self.ds_b = _coerce_dataset(ds_b)
        self.num_classes = num_classes
        self.randomized = randomized
        self.rng = random.Random(seed)

        # Enumerate labels for both sides
        self.labels_a, _ = _enumerate_labels(self.ds_a, map_a, num_classes)
        _, self.buckets_b = _enumerate_labels(self.ds_b, map_b, num_classes)

        # Either verify coverage or drop missing-digit samples from A
        present = {d for d in range(num_classes) if len(self.buckets_b[d]) > 0}
        if not present:
            raise ValueError("Secondary dataset has no digits after mapping.")
        if drop_missing:
            self.idx_a = [i for i, d in enumerate(self.labels_a) if d in present]
        else:
            self.idx_a = list(range(len(self.ds_a)))
            missing = sorted({d for d in set(self.labels_a) if d not in present})
            if missing:
                raise ValueError(f"No samples for digits {missing} in secondary dataset.")

        # per-digit pointer for deterministic cycling
        self.ptr = [0] * num_classes
        for d in range(num_classes):
            self.rng.shuffle(self.buckets_b[d])

    def __len__(self):
        return len(self.idx_a)

    def reseed(self, seed):
        self.rng.seed(seed)
        for d in range(self.num_classes):
            self.rng.shuffle(self.buckets_b[d])
        self.ptr = [0] * self.num_classes

    def _pick_b_index(self, digit):
        bucket = self.buckets_b[digit]
        if self.randomized:
            return self.rng.choice(bucket)
        j = self.ptr[digit]
        self.ptr[digit] = (j + 1) % len(bucket)
        return bucket[j]

    def __getitem__(self, i):
        ai = self.idx_a[i]
        x_aud, y_b_raw = self.ds_a[ai]
        digit = self.labels_a[ai]  # already mapped via map_a

        bj = self._pick_b_index(digit)
        x_img, y_a_raw = self.ds_b[bj]

        return {
            "image": x_img,
            "audio": x_aud,
            "label": int(digit),          # unified 0..9
            "idx_img": int(ai),
            "idx_aud": int(bj),
            "raw_labels": (int(y_a_raw if not torch.is_tensor(y_a_raw) else y_a_raw.item()),
                           int(y_b_raw if not torch.is_tensor(y_b_raw) else y_b_raw.item())),
        }


In [11]:
# 3) Build paired datasets
#    - Anchor on MNIST, map SHD labels via %10
paired_train = LabelMatchedPairDataset(
    ds_a=shd_train_dataset,
    ds_b=mnist_train_dataset,
    map_a=lambda y: y,
    map_b=lambda y: y + random.randint(0, 1) * 10,
    num_classes=20,
    randomized=True,   # set False to cycle deterministically
    seed=123,
)
paired_test = LabelMatchedPairDataset(
    ds_a=shd_test_dataset,
    ds_b=mnist_test_dataset,
    map_a=lambda y: y,
    map_b=lambda y: y + random.randint(0, 1) * 10,
    num_classes=20,
    randomized=True,
)

# 4) DataLoaders for the paired dataset
loader_train = DataLoader(paired_train, batch_size=args.batch_size, shuffle=True,
                                 num_workers=0, pin_memory=False)
loader_eval  = DataLoader(paired_test,  batch_size=args.val_batch_size, shuffle=False,
                                 num_workers=0, pin_memory=False)

In [12]:
len(paired_test)

2264

In [13]:
len(paired_train)

8156

In [14]:
import torch
import torch.nn as nn
import timm

# ---- helpers ---------------------------------------------------------------

def _freeze(module: nn.Module, train_bn: bool = True):
    """Freeze all params; optionally keep BN trainable."""
    for p in module.parameters():
        p.requires_grad = False

def _make_small_input_friendly(m: nn.Module):
    """
    Light tweak for CNNs like ResNet to avoid over-downsampling on 28x28.
    Only applied if attributes exist.
    """
    if hasattr(m, 'conv1') and isinstance(m.conv1, nn.Conv2d):
        in_ch = m.conv1.in_channels
        m.conv1 = nn.Conv2d(in_ch, 64, kernel_size=3, stride=1, padding=1, bias=False)
    if hasattr(m, 'maxpool'):
        m.maxpool = nn.Identity()
    return m

def _build_timm_backbone_mnist(name: str, in_chans: int, pretrained: bool, small_input_friendly: bool):
    """
    Create a timm model that outputs a flattened feature vector [B, D].
    Using num_classes=0 + global_pool='avg' gives pooled features.
    """
    backbone = model_shd = create_model(
    name,
    pretrained=False,
    drop_rate=0.,
    drop_path_rate=0.,
    drop_block_rate=None,
    img_size_h=args.img_size, img_size_w=args.img_size,
    patch_size=args.patch_size, embed_dims=args.dim, num_heads=args.num_heads, mlp_ratios=args.mlp_ratio,
    in_channels=1, num_classes=args.num_classes, qkv_bias=False,
    depths=args.layer, sr_ratios=1,
    T=args.time_step
    )
    if small_input_friendly:
        backbone = _make_small_input_friendly(backbone)
    if pretrained:
        resume_epoch = load_checkpoint(
            model_mnist, './used/mnist_2_384_model_best.pth.tar',
            strict=False)
    # timm models expose .num_features as the pooled feature dim
    feat_dim = getattr(backbone, 'num_features', None)
    if feat_dim is None:
        # Fallback: run a tiny dummy forward (28x28) to infer dim
        backbone.eval()
        with torch.no_grad():
            x = torch.zeros(1, in_chans, 28, 28)
            out = backbone(x)
            feat_dim = out.shape[-1]
    return backbone, feat_dim

def _build_timm_backbone_shd(name: str, in_chans: int, pretrained: bool, small_input_friendly: bool):
    """
    Create a timm model that outputs a flattened feature vector [B, D].
    Using num_classes=0 + global_pool='avg' gives pooled features.
    """
    backbone = model_shd = create_model(
    name,
    pretrained=False,
    drop_rate=0.,
    drop_path_rate=0.,
    drop_block_rate=None,
    img_size_h=args.img_size, img_size_w=args.img_size,
    patch_size=args.patch_size, embed_dims=args.dim, num_heads=args.num_heads, mlp_ratios=args.mlp_ratio,
    in_channels=1, num_classes=args.num_classes, qkv_bias=False,
    depths=args.layer, sr_ratios=1,
    T=args.time_step
    )
    if small_input_friendly:
        backbone = _make_small_input_friendly(backbone)
    if pretrained:
        resume_epoch = load_checkpoint(
            model_shd, './used/shd_model_best.pth.tar',
            strict=False)
    # timm models expose .num_features as the pooled feature dim
    feat_dim = getattr(backbone, 'num_features', None)
    if feat_dim is None:
        # Fallback: run a tiny dummy forward (28x28) to infer dim
        backbone.eval()
        with torch.no_grad():
            x = torch.zeros(1, in_chans, 64, 64)
            out = backbone(x)
            feat_dim = out.shape[-1]
    return backbone, feat_dim

# ---- the dual-backbone concat model ---------------------------------------

class DualTimmConcat(nn.Module):
    """
    Two pre-trained timm backbones.
    forward(x1, x2) -> either concatenated features or final logits.

    Args
    ----
    backbone1, backbone2: str  timm model names (e.g., 'resnet18')
    in_chans1, in_chans2: int  input channels for each backbone
    num_classes: int or None    if set, attach a classifier head; else return concat features
    pretrained: bool            load timm pretrained weights
    freeze_backbones: bool      freeze feature extractors
    train_bn: bool              if freezing, keep BN trainable (often helps)
    small_input_friendly: bool  tweak early downsampling for tiny inputs (e.g., 28x28)
    dropout: float              dropout before classifier
    """
    def __init__(self,
                 backbone1: str,
                 backbone2: str,
                 in_chans1: int = 3,
                 in_chans2: int = 3,
                 num_classes: int = None,
                 pretrained: bool = True,
                 freeze_backbones: bool = False,
                 train_bn: bool = True,
                 small_input_friendly: bool = False,
                 dropout: float = 0.0):
        super().__init__()

        self.m1, d1 = _build_timm_backbone_mnist(backbone1, in_chans1, pretrained, small_input_friendly)
        self.m2, d2 = _build_timm_backbone_shd(backbone2, in_chans2, pretrained, small_input_friendly)
        self.out_dim = d1 + d2
        self.out_dim2 = int((d1 + d2)/2)

        if freeze_backbones:
            _freeze(self.m1, train_bn=train_bn)
            _freeze(self.m2, train_bn=train_bn)

        self.concat_only = (num_classes is None)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.head = None if self.concat_only else nn.Linear(self.out_dim, self.out_dim2)
        self.head2 = None if self.concat_only else nn.Linear(self.out_dim2, num_classes)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor, return_features: bool = False):
        """
        x1 -> backbone1, x2 -> backbone2.
        Returns:
          - if self.concat_only or return_features: concatenated features [B, D1+D2]
          - else: logits [B, num_classes]
        """
        z1 = self.m1(x1)   # [B, D1]
        z2 = self.m2(x2)   # [B, D2]
        z  = torch.cat([z1, z2], dim=-1)
        if self.concat_only or return_features:
            return z
        z = self.dropout(z)
        z = self.head(z)
        z = self.head2(z)
        return z


In [15]:
# ---- example wiring with your paired loader --------------------------------
if __name__ == "__main__":
    # Example: MNIST image (1x28x28) + SHD spectrogram (1x64x64)
    # Choose backbones that work well on small inputs; resnet18 is fine after small_input_friendly tweak.
    model = DualTimmConcat(
        backbone1='sdt', in_chans1=1,       # MNIST grayscale
        backbone2='sdt', in_chans2=1,       # SHD spectrogram (1 channel)
        num_classes=20,                           # final classifier; set None to return concat features
        pretrained=False,
        freeze_backbones=True,
        small_input_friendly=False,               # avoids heavy downsampling on tiny inputs
        dropout=0.0
    )

    # Dummy batch shapes (match your collate output):
    #x_img = torch.randn(args.batch_size, 1, 28, 28)
    #x_aud = torch.randn(args.batch_size, 1, 64, 64)     # if your SHD is [B,T,F], convert to [B,1,H,W] upstream
    #logits = model(x_img, x_aud)                # [8, 10]
    #print("logits:", logits.shape)

In [16]:
for q, p in model.named_parameters():
     p.requires_grad = True

In [17]:
for q, p in model.named_parameters():
     if p.requires_grad:
          print(q)

m1.patch_embed.proj_conv.weight
m1.patch_embed.proj_bn.weight
m1.patch_embed.proj_bn.bias
m1.patch_embed.proj_conv1.weight
m1.patch_embed.proj_bn1.weight
m1.patch_embed.proj_bn1.bias
m1.patch_embed.proj_conv2.weight
m1.patch_embed.proj_bn2.weight
m1.patch_embed.proj_bn2.bias
m1.patch_embed.proj_conv3.weight
m1.patch_embed.proj_bn3.weight
m1.patch_embed.proj_bn3.bias
m1.patch_embed.rpe_conv.weight
m1.patch_embed.rpe_bn.weight
m1.patch_embed.rpe_bn.bias
m1.block.0.attn.q_conv.weight
m1.block.0.attn.q_bn.weight
m1.block.0.attn.q_bn.bias
m1.block.0.attn.k_conv.weight
m1.block.0.attn.k_bn.weight
m1.block.0.attn.k_bn.bias
m1.block.0.attn.v_conv.weight
m1.block.0.attn.v_bn.weight
m1.block.0.attn.v_bn.bias
m1.block.0.attn.talking_heads.weight
m1.block.0.attn.proj_conv.weight
m1.block.0.attn.proj_conv.bias
m1.block.0.attn.proj_bn.weight
m1.block.0.attn.proj_bn.bias
m1.block.0.mlp.fc1_conv.weight
m1.block.0.mlp.fc1_conv.bias
m1.block.0.mlp.fc1_bn.weight
m1.block.0.mlp.fc1_bn.bias
m1.block.0.mlp.

In [18]:
print("Creating model")
n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"number of params: {n_parameters}")

Creating model
number of params: 5264340


In [19]:
model.cuda()

DualTimmConcat(
  (m1): SpikeDrivenTransformer(
    (patch_embed): MS_SPS(
      (proj_conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (proj_bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (proj_lif): MultiStepLIFNode(
        v_threshold=1.0, v_reset=0.0, detach_reset=True, tau=2.0, backend=torch
        (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
      )
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (proj_conv1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (proj_bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (proj_lif1): MultiStepLIFNode(
        v_threshold=1.0, v_reset=0.0, detach_reset=True, tau=2.0, backend=torch
        (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
      )
      (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=1, dilat

In [20]:
device = torch.device('cuda', args.local_rank)

In [21]:
if args.num_classes is None:
    assert hasattr(model, 'num_classes'), 'Model must have `num_classes` attr if not set on cmd line/config.'
    args.num_classes = model.num_classes  # FIXME handle model default vs config num_classes more elegantly

if args.local_rank == 0:
    _logger.info(
        f'Model {safe_model_name(args.model)} created, param count:{sum([m.numel() for m in model.parameters()])}')

data_config = resolve_data_config(vars(args), model=model, verbose=args.local_rank == 0)

# setup augmentation batch splits for contrastive loss or split bn
num_aug_splits = 0
if args.aug_splits > 0:
    assert args.aug_splits > 1, 'A split of 1 makes no sense'
    num_aug_splits = args.aug_splits

# enable split bn (separate bn stats per batch-portion)
if args.split_bn:
    assert num_aug_splits > 1 or args.resplit
    model = convert_splitbn_model(model, max(num_aug_splits, 2))

# move model to GPU, enable channels last layout if set
model.to(device)
if args.channels_last:
    model = model.to(memory_format=torch.channels_last)

# setup synchronized BatchNorm for distributed training
if args.distributed and args.sync_bn:
    assert not args.split_bn
    if has_apex and use_amp != 'native':
        # Apex SyncBN preferred unless native amp is activated
        model = convert_syncbn_model(model)
    else:
        model = torch.nn.SyncBatchNorm.convert_sync_batchnorm(model)
    if args.local_rank == 0:
        _logger.info(
            'Converted model to use Synchronized BatchNorm. WARNING: You may have issues if using '
            'zero initialized BN layers (enabled by default for ResNets) while sync-bn enabled.')

if args.torchscript:
    assert not use_amp == 'apex', 'Cannot use APEX AMP with torchscripted model'
    assert not args.sync_bn, 'Cannot use SyncBatchNorm with torchscripted model'
    model = torch.jit.script(model)

optimizer = create_optimizer_v2(model, **optimizer_kwargs(cfg=args))

# setup automatic mixed-precision (AMP) loss scaling and op casting
amp_autocast = suppress  # do nothing
loss_scaler = None
if use_amp == 'apex':
    model, optimizer = amp.initialize(model, optimizer, opt_level='O1')
    loss_scaler = ApexScaler()
    if args.local_rank == 0:
        _logger.info('Using NVIDIA APEX AMP. Training in mixed precision.')
elif use_amp == 'native':
    amp_autocast = torch.cuda.amp.autocast
    loss_scaler = NativeScaler()
    if args.local_rank == 0:
        _logger.info('Using native Torch AMP. Training in mixed precision.')
else:
    if args.local_rank == 0:
        _logger.info('AMP not enabled. Training in float32.')

INFO:train:Model sdt created, param count:5264340
Model sdt created, param count:5264340
INFO:timm.data.config:Data processing configuration for current model + dataset:
Data processing configuration for current model + dataset:
INFO:timm.data.config:	input_size: (1, 64, 64)
	input_size: (1, 64, 64)
INFO:timm.data.config:	interpolation: bicubic
	interpolation: bicubic
INFO:timm.data.config:	mean: (0.485, 0.456, 0.406)
	mean: (0.485, 0.456, 0.406)
INFO:timm.data.config:	std: (0.229, 0.224, 0.225)
	std: (0.229, 0.224, 0.225)
INFO:timm.data.config:	crop_pct: 0.875
	crop_pct: 0.875
INFO:train:Using native Torch AMP. Training in mixed precision.
Using native Torch AMP. Training in mixed precision.


In [22]:
# setup exponential moving average of model weights, SWA could be used here too
model_ema = None
if args.model_ema:
    # Important to create EMA model after cuda(), DP wrapper, and AMP but before SyncBN and DDP wrapper
    model_ema = ModelEmaV2(
        model, decay=args.model_ema_decay, device='cpu' if args.model_ema_force_cpu else None)

# setup distributed training
# setup distributed training
if args.distributed:
    if has_apex and use_amp != 'native':
        # Apex DDP preferred unless native amp is activated
        if args.local_rank == 0:
            _logger.info("Using NVIDIA APEX DistributedDataParallel.")
        model = ApexDDP(model, delay_allreduce=True)
    else:
        if args.local_rank == 0:
            _logger.info("Using native Torch DistributedDataParallel.")
        model = NativeDDP(model, device_ids=[args.local_rank],find_unused_parameters=True)  # can use device str in Torch >= 1.1
    # NOTE: EMA model does not need to be wrapped by DDP

# setup learning rate schedule and starting epoch
lr_scheduler, num_epochs = create_scheduler(args, optimizer)
start_epoch = 0
if args.start_epoch is not None:
    # a specified start_epoch will always override the resume epoch
    start_epoch = args.start_epoch
if lr_scheduler is not None and start_epoch > 0:
    lr_scheduler.step(start_epoch)

if args.local_rank == 0:
    _logger.info('Scheduled epochs: {}'.format(num_epochs))

# setup mixup / cutmix
collate_fn = None
mixup_fn = None
mixup_active = args.mixup > 0 or args.cutmix > 0. or args.cutmix_minmax is not None
if mixup_active:
    mixup_args = dict(
        mixup_alpha=args.mixup, cutmix_alpha=args.cutmix, cutmix_minmax=args.cutmix_minmax,
        prob=args.mixup_prob, switch_prob=args.mixup_switch_prob, mode=args.mixup_mode,
        label_smoothing=args.smoothing, num_classes=args.num_classes)
    if args.prefetcher:
        assert not num_aug_splits  # collate conflict (need to support deinterleaving in collate mixup)
        collate_fn = FastCollateMixup(**mixup_args)
    else:
        mixup_fn = Mixup(**mixup_args)


# setup loss function
# if args.jsd:
#     assert num_aug_splits > 1  # JSD only valid with aug splits set
#     train_loss_fn = JsdCrossEntropy(num_splits=num_aug_splits, smoothing=args.smoothing).cuda()
# elif mixup_active:
#     # smoothing is handled with mixup target transform
#     train_loss_fn = SoftTargetCrossEntropy().cuda()
# elif args.smoothing:
#     train_loss_fn = LabelSmoothingCrossEntropy(smoothing=args.smoothing).cuda()
# else:
train_loss_fn = nn.CrossEntropyLoss().cuda()
validate_loss_fn = nn.CrossEntropyLoss().cuda()

# setup checkpoint saver and eval metric tracking
eval_metric = args.eval_metric
best_metric = None
best_epoch = None
saver = None
output_dir = None
if args.rank == 0:
    if args.experiment:
        exp_name = args.experiment
    else:
        exp_name = '-'.join([
            datetime.now().strftime("%Y%m%d-%H%M%S"),
            safe_model_name(args.model),
            str(data_config['input_size'][-1])
        ])
    output_dir = get_outdir(args.output if args.output else './output/train', exp_name)
    decreasing = True if eval_metric == 'loss' else False
    saver = CheckpointSaver(
        model=model, optimizer=optimizer, args=args, model_ema=model_ema, amp_scaler=loss_scaler,
        checkpoint_dir=output_dir, recovery_dir=output_dir, decreasing=decreasing, max_history=args.checkpoint_hist)
    with open(os.path.join(output_dir, 'args.yaml'), 'w') as f:
        f.write(args_text)

INFO:train:Scheduled epochs: 310
Scheduled epochs: 310


## Validate Multimodal (Acc. 98.54~)

In [28]:
# optionally resume from a checkpoint
args.resume = "./used/sdt_shd_mnist_model_98.58.tar"
resume_epoch = None
if args.resume:
    resume_epoch = load_checkpoint(
        model, args.resume,
        strict=False)

INFO:timm.models.helpers:Loaded state_dict from checkpoint './used/sdt_shd_mnist_model_98.58.tar'
Loaded state_dict from checkpoint './used/sdt_shd_mnist_model_98.58.tar'


In [29]:
amp_autocast_test = torch.cuda.amp.autocast(dtype=torch.float16)
eval_metrics = validate_multimodal(model, loader_eval, validate_loss_fn, args, amp_autocast=amp_autocast_test)

INFO:train:Test: [   0/70]  Time: 0.252 (0.252)  Loss:  0.1214 (0.1214)  Acc@1: 93.7500 (93.7500)  Acc@5: 100.0000 (100.0000)
Test: [   0/70]  Time: 0.252 (0.252)  Loss:  0.1214 (0.1214)  Acc@1: 93.7500 (93.7500)  Acc@5: 100.0000 (100.0000)
INFO:train:Test: [  70/70]  Time: 0.156 (0.211)  Loss:  0.0059 (0.0439)  Acc@1: 100.0000 (98.6749)  Acc@5: 100.0000 (100.0000)
Test: [  70/70]  Time: 0.156 (0.211)  Loss:  0.0059 (0.0439)  Acc@1: 100.0000 (98.6749)  Acc@5: 100.0000 (100.0000)


In [30]:
def validate_multimodal_half(model, loader, loss_fn, args, amp_autocast=suppress, log_suffix=''):
    batch_time_m = AverageMeter()
    losses_m = AverageMeter()
    top1_m = AverageMeter()
    top5_m = AverageMeter()

    model = model.cuda().half()
    model.eval()

    end = time.time()
    last_idx = len(loader) - 1
    with torch.no_grad():
        for batch_idx, y in enumerate(loader):
            image = y['image']
            audio = y['audio']
            target = y['label']
            last_batch = batch_idx == last_idx
            image, audio, target = image.cuda().half(), audio.cuda().half(), target.cuda()
            if args.channels_last:
                input = input.contiguous(memory_format=torch.channels_last)

            output = model(image, audio)
            if isinstance(output, (tuple, list)):
                output = output[0]

            # augmentation reduction
            reduce_factor = args.tta
            if reduce_factor > 1:
                output = output.unfold(0, reduce_factor, reduce_factor).mean(dim=2)
                target = target[0:target.size(0):reduce_factor]
            loss = loss_fn(output, target)
            functional.reset_net(model)

            acc1, acc5 = accuracy(output, target, topk=(1, 5))

            if args.distributed:
                reduced_loss = reduce_tensor(loss.data, args.world_size)
                acc1 = reduce_tensor(acc1, args.world_size)
                acc5 = reduce_tensor(acc5, args.world_size)
            else:
                reduced_loss = loss.data

            torch.cuda.synchronize()

            losses_m.update(reduced_loss.item(), image.size(0))
            top1_m.update(acc1.item(), output.size(0))
            top5_m.update(acc5.item(), output.size(0))

            batch_time_m.update(time.time() - end)
            end = time.time()
            if args.local_rank == 0 and (last_batch or batch_idx % args.log_interval == 0):
                log_name = 'Test' + log_suffix
                _logger.info(
                    '{0}: [{1:>4d}/{2}]  '
                    'Time: {batch_time.val:.3f} ({batch_time.avg:.3f})  '
                    'Loss: {loss.val:>7.4f} ({loss.avg:>6.4f})  '
                    'Acc@1: {top1.val:>7.4f} ({top1.avg:>7.4f})  '
                    'Acc@5: {top5.val:>7.4f} ({top5.avg:>7.4f})'.format(
                        log_name, batch_idx, last_idx, batch_time=batch_time_m,
                        loss=losses_m, top1=top1_m, top5=top5_m))

    metrics = OrderedDict([('loss', losses_m.avg), ('top1', top1_m.avg), ('top5', top5_m.avg)])

    return metrics

In [31]:
eval_metrics = validate_multimodal_half(model, loader_eval, validate_loss_fn, args, amp_autocast=amp_autocast)

INFO:train:Test: [   0/70]  Time: 0.225 (0.225)  Loss:  0.1354 (0.1354)  Acc@1: 93.7500 (93.7500)  Acc@5: 100.0000 (100.0000)
Test: [   0/70]  Time: 0.225 (0.225)  Loss:  0.1354 (0.1354)  Acc@1: 93.7500 (93.7500)  Acc@5: 100.0000 (100.0000)
INFO:train:Test: [  70/70]  Time: 0.156 (0.206)  Loss:  0.0045 (0.0426)  Acc@1: 100.0000 (98.6749)  Acc@5: 100.0000 (100.0000)
Test: [  70/70]  Time: 0.156 (0.206)  Loss:  0.0045 (0.0426)  Acc@1: 100.0000 (98.6749)  Acc@5: 100.0000 (100.0000)


In [27]:
assert False

AssertionError: 

In [23]:
for epoch in range(start_epoch, num_epochs):
    if args.distributed and hasattr(loader_train.sampler, 'set_epoch'):
        loader_train.sampler.set_epoch(epoch)

    train_metrics = train_one_epoch(
        epoch, model, loader_train, optimizer, train_loss_fn, args,
        lr_scheduler=lr_scheduler, saver=saver, output_dir=output_dir,
        amp_autocast=amp_autocast, loss_scaler=loss_scaler, model_ema=model_ema, mixup_fn=mixup_fn)

    if args.distributed and args.dist_bn in ('broadcast', 'reduce'):
        if args.local_rank == 0:
            _logger.info("Distributing BatchNorm running means and vars")
        distribute_bn(model, args.world_size, args.dist_bn == 'reduce')

    eval_metrics = validate_multimodal(model, loader_eval, validate_loss_fn, args, amp_autocast=amp_autocast)

    if model_ema is not None and not args.model_ema_force_cpu:
        if args.distributed and args.dist_bn in ('broadcast', 'reduce'):
            distribute_bn(model_ema, args.world_size, args.dist_bn == 'reduce')
        ema_eval_metrics = validate_multimodal(
            model_ema.module, loader_eval, validate_loss_fn, args, amp_autocast=amp_autocast,
            log_suffix=' (EMA)')
        eval_metrics = ema_eval_metrics

    if lr_scheduler is not None:
        # step LR for next epoch
        lr_scheduler.step(epoch + 1, eval_metrics[eval_metric])

    if output_dir is not None:
        update_summary(
            epoch, train_metrics, eval_metrics, os.path.join(output_dir, 'summary.csv'),
            write_header=best_metric is None, log_wandb=args.log_wandb and has_wandb)

    if saver is not None:
        # save proper checkpoint with eval metric
        save_metric = eval_metrics[eval_metric]
        best_metric, best_epoch = saver.save_checkpoint(epoch, metric=save_metric)
        _logger.info('*** Best metric: {0} (epoch {1})'.format(best_metric, best_epoch))

INFO:train:Train: 0 [   0/255 (  0%)]  Loss:  2.988464 (2.9885)  Time: 3.244s,    9.86/s  (3.244s,    9.86/s)  LR: 1.000e-05  Data: 0.274 (0.274)
Train: 0 [   0/255 (  0%)]  Loss:  2.988464 (2.9885)  Time: 3.244s,    9.86/s  (3.244s,    9.86/s)  LR: 1.000e-05  Data: 0.274 (0.274)
INFO:train:Train: 0 [ 254/255 (100%)]  Loss:  2.712612 (2.8829)  Time: 2.260s,   12.39/s  (0.408s,   68.62/s)  LR: 1.000e-05  Data: 0.197 (0.231)
Train: 0 [ 254/255 (100%)]  Loss:  2.712612 (2.8829)  Time: 2.260s,   12.39/s  (0.408s,   68.62/s)  LR: 1.000e-05  Data: 0.197 (0.231)
INFO:train:Test: [   0/70]  Time: 0.282 (0.282)  Loss:  2.6484 (2.6484)  Acc@1: 31.2500 (31.2500)  Acc@5: 81.2500 (81.2500)
Test: [   0/70]  Time: 0.282 (0.282)  Loss:  2.6484 (2.6484)  Acc@1: 31.2500 (31.2500)  Acc@5: 81.2500 (81.2500)
INFO:train:Test: [  70/70]  Time: 0.664 (0.213)  Loss:  2.6094 (2.6759)  Acc@1: 29.1667 (28.1360)  Acc@5: 70.8333 (75.3534)
Test: [  70/70]  Time: 0.664 (0.213)  Loss:  2.6094 (2.6759)  Acc@1: 29.1667 

In [26]:
assert False

AssertionError: 

## Validate MNIST (Acc. 99.39)

In [ ]:
model_mnist = create_model(
    'spikformer',
    pretrained=False,
    drop_rate=0.,
    drop_path_rate=0.,
    drop_block_rate=None,
    img_size_h=28, img_size_w=28,
    patch_size=4, embed_dims=384, num_heads=12, mlp_ratios=4,
    in_channels=1, num_classes=10, qkv_bias=False,
    depths=2, sr_ratios=1,
    T=4, activate_classifier=True
    )
resume_epoch = load_checkpoint(
    model_mnist, './used/mnist_2_384_model_best.pth.tar',
    strict=False)

INFO:timm.models.helpers:Loaded state_dict from checkpoint './used/mnist_2_384_model_best.pth.tar'
Loaded state_dict from checkpoint './used/mnist_2_384_model_best.pth.tar'


In [ ]:
model_mnist.cuda()

Spikformer(
  (patch_embed): SPS(
    (proj_conv): Conv2d(1, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (proj_bn): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (proj_lif): MultiStepLIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=True, tau=2.0, backend=torch
      (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
    )
    (proj_conv1): Conv2d(48, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (proj_bn1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (proj_lif1): MultiStepLIFNode(
      v_threshold=1.0, v_reset=0.0, detach_reset=True, tau=2.0, backend=torch
      (surrogate_function): Sigmoid(alpha=4.0, spiking=True)
    )
    (proj_conv2): Conv2d(96, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (proj_bn2): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (proj_lif2): MultiStepLIFNod

In [ ]:
from loader import create_loader
loader_eval_mnist = create_loader(
    mnist_test_dataset,
    input_size=(1, 28, 28),
    batch_size=args.batch_size,
    is_training=False,
    use_prefetcher=True,
    mean=mnist_mean,
    std=mnist_std,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
def validate_mnist(model, loader, loss_fn, args, amp_autocast=suppress, log_suffix=''):
    batch_time_m = AverageMeter()
    losses_m = AverageMeter()
    top1_m = AverageMeter()
    top5_m = AverageMeter()

    model.eval()

    end = time.time()
    last_idx = len(loader) - 1
    with torch.no_grad():
        for batch_idx, (input, target) in enumerate(loader):
            last_batch = batch_idx == last_idx
            if not args.prefetcher:
                input = input.cuda()
                target = target.cuda()
            if args.channels_last:
                input = input.contiguous(memory_format=torch.channels_last)

            with amp_autocast():
                output = model(input)
            if isinstance(output, (tuple, list)):
                output = output[0]

            # augmentation reduction
            reduce_factor = args.tta
            if reduce_factor > 1:
                output = output.unfold(0, reduce_factor, reduce_factor).mean(dim=2)
                target = target[0:target.size(0):reduce_factor]

            loss = loss_fn(output, target)
            functional.reset_net(model)

            acc1, acc5 = accuracy(output, target, topk=(1, 5))

            if args.distributed:
                reduced_loss = reduce_tensor(loss.data, args.world_size)
                acc1 = reduce_tensor(acc1, args.world_size)
                acc5 = reduce_tensor(acc5, args.world_size)
            else:
                reduced_loss = loss.data

            torch.cuda.synchronize()

            losses_m.update(reduced_loss.item(), input.size(0))
            top1_m.update(acc1.item(), output.size(0))
            top5_m.update(acc5.item(), output.size(0))

            batch_time_m.update(time.time() - end)
            end = time.time()
            if args.local_rank == 0 and (last_batch or batch_idx % args.log_interval == 0):
                log_name = 'Test' + log_suffix
                _logger.info(
                    '{0}: [{1:>4d}/{2}]  '
                    'Time: {batch_time.val:.3f} ({batch_time.avg:.3f})  '
                    'Loss: {loss.val:>7.4f} ({loss.avg:>6.4f})  '
                    'Acc@1: {top1.val:>7.4f} ({top1.avg:>7.4f})  '
                    'Acc@5: {top5.val:>7.4f} ({top5.avg:>7.4f})'.format(
                        log_name, batch_idx, last_idx, batch_time=batch_time_m,
                        loss=losses_m, top1=top1_m, top5=top5_m))

    metrics = OrderedDict([('loss', losses_m.avg), ('top1', top1_m.avg), ('top5', top5_m.avg)])

    return metrics

In [ ]:
eval_metrics = validate_mnist(model_mnist, loader_eval_mnist, validate_loss_fn, args, amp_autocast=amp_autocast)

INFO:train:Test: [   0/312]  Time: 0.083 (0.083)  Loss:  0.0010 (0.0010)  Acc@1: 100.0000 (100.0000)  Acc@5: 100.0000 (100.0000)
Test: [   0/312]  Time: 0.083 (0.083)  Loss:  0.0010 (0.0010)  Acc@1: 100.0000 (100.0000)  Acc@5: 100.0000 (100.0000)
INFO:train:Test: [ 312/312]  Time: 0.112 (0.018)  Loss:  0.0001 (0.0228)  Acc@1: 100.0000 (99.3900)  Acc@5: 100.0000 (99.9900)
Test: [ 312/312]  Time: 0.112 (0.018)  Loss:  0.0001 (0.0228)  Acc@1: 100.0000 (99.3900)  Acc@5: 100.0000 (99.9900)
